# Speaker Verification using ResNet

This notebook implements a text-independent speaker verification system using:

- LibriSpeech train-clean-100
- 50,000 balanced speaker pairs
- 3-second random crop or padding
- Log-Mel Spectrogram (80 mel bins)
- ResNet backbone

---

## 1. Dataset Setup

In [1]:
import os
import pandas as pd
import numpy as np

# Dataset root
DATASET_ROOT = "/kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset"

# Inner audio folder (second train-clean-100)
BASE_AUDIO_DIR = os.path.join(DATASET_ROOT, "train-clean-100", "train-clean-100")

# CSV path
CSV_PATH = os.path.join(DATASET_ROOT, "train_pairs.csv")

print("BASE_AUDIO_DIR:", BASE_AUDIO_DIR)
print("CSV_PATH:", CSV_PATH)

BASE_AUDIO_DIR: /kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset/train-clean-100/train-clean-100
CSV_PATH: /kaggle/input/datasets/tahmidulislamomi/ml-project-openslr-dataset/train_pairs.csv


## 2. Load and Inspect Training Data

Load the training CSV file and display the first few rows along with the label distribution to confirm the dataset is balanced.

In [2]:
df = pd.read_csv(CSV_PATH)

print("First 5 rows:")
display(df.head())

print("Total rows:", len(df))
print("Label distribution:")
print(df["label"].value_counts())

First 5 rows:


,audio_path_1,audio_path_2,label
0,train-clean-100/311/124404/311-124404-0059.flac,train-clean-100/311/124404/311-124404-0009.flac,1
1,train-clean-100/1455/134435/1455-134435-0051.flac,train-clean-100/1455/134435/1455-134435-0058.flac,1
2,train-clean-100/4397/15668/4397-15668-0028.flac,train-clean-100/4397/15678/4397-15678-0002.flac,1
3,train-clean-100/2514/149482/2514-149482-0028.flac,train-clean-100/2514/149482/2514-149482-0038.flac,1
4,train-clean-100/669/129074/669-129074-0021.flac,train-clean-100/669/129061/669-129061-0026.flac,1


Total rows: 50000
Label distribution:
label
1    25000
0    25000
Name: count, dtype: int64


## 3. Resolve Absolute Audio Paths

The CSV contains relative paths prefixed with `train-clean-100/`. `to_absolute_path()` strips that prefix and joins with the true Kaggle dataset root so that `soundfile` can locate each `.flac` file.

In [3]:
def to_absolute_path(rel_path):
    # Remove first occurrence of train-clean-100/
    cleaned = rel_path.replace("train-clean-100/", "", 1)
    return os.path.join(BASE_AUDIO_DIR, cleaned)

df["path1_abs"] = df["audio_path_1"].apply(to_absolute_path)
df["path2_abs"] = df["audio_path_2"].apply(to_absolute_path)

display(df[["path1_abs", "path2_abs", "label"]].head())

,path1_abs,path2_abs,label
0,/kaggle/input/datasets/tahmidulislamomi/ml-pro...,/kaggle/input/datasets/tahmidulislamomi/ml-pro...,1
1,/kaggle/input/datasets/tahmidulislamomi/ml-pro...,/kaggle/input/datasets/tahmidulislamomi/ml-pro...,1
2,/kaggle/input/datasets/tahmidulislamomi/ml-pro...,/kaggle/input/datasets/tahmidulislamomi/ml-pro...,1
3,/kaggle/input/datasets/tahmidulislamomi/ml-pro...,/kaggle/input/datasets/tahmidulislamomi/ml-pro...,1
4,/kaggle/input/datasets/tahmidulislamomi/ml-pro...,/kaggle/input/datasets/tahmidulislamomi/ml-pro...,1


In [4]:
missing_1 = (~df["path1_abs"].apply(os.path.exists)).sum()
missing_2 = (~df["path2_abs"].apply(os.path.exists)).sum()

print("Missing path1 files:", missing_1)
print("Missing path2 files:", missing_2)

Missing path1 files: 0
Missing path2 files: 0


## 4. Audio Sample Verification

Read one audio pair from disk to confirm that the resolved paths are valid and that the sample rates match the expected 16 kHz.

In [5]:
import soundfile as sf

sample_row = df.iloc[0]

audio1, sr1 = sf.read(sample_row["path1_abs"])
audio2, sr2 = sf.read(sample_row["path2_abs"])

print("Audio1 shape:", audio1.shape, "Sample rate:", sr1)
print("Audio2 shape:", audio2.shape, "Sample rate:", sr2)

Audio1 shape: (228160,) Sample rate: 16000
Audio2 shape: (174560,) Sample rate: 16000


In [6]:
import numpy as np

## Standardize Audio Length (3 Seconds)

- Target sample rate: 16 kHz  
- Target duration: 3 seconds (48,000 samples)

Function `crop_or_pad()`:
- If audio is longer → random crop to 3 seconds  
- If audio is shorter → zero pad to 3 seconds  
- Ensures fixed-length input for neural network training

In [7]:
TARGET_SR = 16000
TARGET_DURATION = 3  # seconds
TARGET_LENGTH = TARGET_SR * TARGET_DURATION  # 48000


def crop_or_pad(audio):
    """
    Ensures audio is exactly 3 seconds (48000 samples).
    - If longer → random crop
    - If shorter → zero pad
    """
    length = len(audio)

    # If longer → random crop
    if length > TARGET_LENGTH:
        start = np.random.randint(0, length - TARGET_LENGTH)
        audio = audio[start:start + TARGET_LENGTH]

    # If shorter → pad with zeros
    elif length < TARGET_LENGTH:
        pad_amount = TARGET_LENGTH - length
        audio = np.pad(audio, (0, pad_amount), mode='constant')

    return audio

In [8]:
audio1_fixed = crop_or_pad(audio1)
audio2_fixed = crop_or_pad(audio2)

print(audio1_fixed.shape)  # should be (48000,)
print(audio2_fixed.shape)

(48000,)
(48000,)


In [9]:
import torchaudio
import torch
import torchaudio.transforms as T

## Create Log-Mel Spectrogram Transform

- Sample rate: 16 kHz  
- Window size: 25 ms (n_fft = 400)  
- Hop length: 10 ms (160 samples)  
- Number of Mel bins: 80  

`MelSpectrogram` converts waveform → Mel-frequency representation.  
`AmplitudeToDB` converts amplitude to log scale for stable training.

In [10]:
mel_transform = T.MelSpectrogram(
    sample_rate=16000,
    n_fft=400,              # 25 ms window (16000 * 0.025 = 400)
    hop_length=160,         # 10 ms stride (16000 * 0.01 = 160)
    n_mels=80
)

amplitude_to_db = T.AmplitudeToDB()

## Convert Audio to Log-Mel Spectrogram

Steps:
- Convert NumPy audio → Torch tensor  
- Add channel dimension (1, 48000)  
- Apply MelSpectrogram transform  
- Convert to log scale (dB)

Final output shape:
(1, 80, ~300) → Image-like input for CNN/ResNet

In [11]:
# Convert numpy → torch tensor
audio_tensor = torch.tensor(audio1_fixed).float()

# Add channel dimension (1, 48000)
audio_tensor = audio_tensor.unsqueeze(0)

# Compute Mel spectrogram
mel_spec = mel_transform(audio_tensor)

# Convert to log scale
log_mel_spec = amplitude_to_db(mel_spec)

print("Log-Mel shape:", log_mel_spec.shape)

Log-Mel shape: torch.Size([1, 80, 301])


## 5. Custom Dataset Class

`SpeakerPairDataset` handles the full preprocessing pipeline on the fly:
- Reads raw `.flac` audio with `soundfile`
- Converts stereo to mono if needed
- Applies `crop_or_pad()` to fix length at 3 seconds
- Computes the Log-Mel Spectrogram
- Returns `(mel1, mel2, label)` per pair

In [12]:
from torch.utils.data import Dataset
import soundfile as sf

In [13]:
class SpeakerPairDataset(Dataset):
    def __init__(self, dataframe, mel_transform, amplitude_to_db):
        self.df = dataframe
        self.mel_transform = mel_transform
        self.amplitude_to_db = amplitude_to_db

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        audio, sr = sf.read(path)

        # Ensure mono
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)

        audio = crop_or_pad(audio)

        audio = torch.tensor(audio).float().unsqueeze(0)

        mel = self.mel_transform(audio)
        log_mel = self.amplitude_to_db(mel)

        return log_mel

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        mel1 = self.load_audio(row["path1_abs"])
        mel2 = self.load_audio(row["path2_abs"])

        label = torch.tensor(row["label"]).float()

        return mel1, mel2, label

## 6. Instantiate Training Dataset

Create a `SpeakerPairDataset` from the full 50,000-pair training DataFrame.

In [14]:
dataset = SpeakerPairDataset(df, mel_transform, amplitude_to_db)

print("Dataset size:", len(dataset))

Dataset size: 50000


In [15]:
mel1, mel2, label = dataset[0]

print("Mel1 shape:", mel1.shape)
print("Mel2 shape:", mel2.shape)
print("Label:", label)

Mel1 shape: torch.Size([1, 80, 301])
Mel2 shape: torch.Size([1, 80, 301])
Label: tensor(1.)


## 7. DataLoader Setup — Training

`DataLoader` wraps the dataset for efficient batched, multi-worker loading.
- `batch_size = 16` — number of pairs per batch
- `shuffle = True` — randomize order every epoch
- `pin_memory = True` — speeds up GPU data transfer

In [16]:
from torch.utils.data import DataLoader

batch_size = 16

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print("DataLoader ready")

DataLoader ready


In [17]:
batch_mel1, batch_mel2, batch_labels = next(iter(dataloader))

print("Batch mel1:", batch_mel1.shape)
print("Batch mel2:", batch_mel2.shape)
print("Batch labels:", batch_labels.shape)

Batch mel1: torch.Size([16, 1, 80, 301])
Batch mel2: torch.Size([16, 1, 80, 301])
Batch labels: torch.Size([16])


## 8. Model Architecture — ResNetSpeaker

`ResNetSpeaker` is a Siamese network built on ResNet-18:
- **Input conv** modified from 3-channel RGB to 1-channel (grayscale Log-Mel Spectrogram)
- **Classification head** replaced with a 256-dimensional embedding layer
- **L2 normalization** applied to embeddings, enabling direct cosine similarity comparison

In [18]:
import torch.nn as nn
import torchvision.models as models

In [19]:
class ResNetSpeaker(nn.Module):
    def __init__(self, embedding_dim=256):
        super().__init__()

        # Load ResNet18 
        self.backbone = models.resnet18(weights=None)

        # Modify first conv layer to accept 1
        self.backbone.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        # Remove final classification layer
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        # Add embedding layer
        self.embedding = nn.Linear(in_features, embedding_dim)

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embedding(features)

        # Normalize embedding (important for cosine similarity)
        embedding = nn.functional.normalize(embedding, p=2, dim=1)

        return embedding

## 9. Initialize Model on Device

Detect and use GPU (CUDA) if available, otherwise fall back to CPU. Instantiate and print the full `ResNetSpeaker` architecture.

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNetSpeaker(embedding_dim=256).to(device)

print(model)

ResNetSpeaker(
  (backbone): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tr

In [21]:
batch_mel1, batch_mel2, batch_labels = next(iter(dataloader))

batch_mel1 = batch_mel1.to(device)

embeddings = model(batch_mel1)

print("Embedding shape:", embeddings.shape)

Embedding shape: torch.Size([16, 256])


In [22]:
import torch.nn.functional as F

## 10. Contrastive Loss Function

`ContrastiveLoss` trains the model to:
- **Pull** same-speaker embeddings *closer* together (label = 1)
- **Push** different-speaker embeddings *further apart* (label = 0)

Distance is measured using **Euclidean distance** between the two L2-normalized embedding vectors.

In [23]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        # Euclidean distance
        distance = F.pairwise_distance(emb1, emb2)

        # Positive loss (label=1)
        pos_loss = label * torch.pow(distance, 2)

        # Negative loss (label=0)
        neg_loss = (1 - label) * torch.pow(torch.clamp(self.margin - distance, min=0.0), 2)

        loss = torch.mean(pos_loss + neg_loss)
        return loss

## 11. Training Configuration

Reinitialize the model (clean weights), define the contrastive loss criterion with `margin = 1.0`, and configure the **Adam optimizer** with a learning rate of `1e-3`.

In [24]:
model = ResNetSpeaker(256).to(device)

criterion = ContrastiveLoss(margin=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 12. Training Loop

Train for **30 epochs** over the full 50,000 training pairs.
- Each batch: forward pass → compute contrastive loss → backpropagation → optimizer step
- Live loss is displayed in the `tqdm` progress bar
- Average epoch loss is printed at the end of each epoch

In [25]:
from tqdm import tqdm

num_epochs = 30
print_every = 50   # print every 50 batches

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    progress_bar = tqdm(enumerate(dataloader), 
                        total=len(dataloader),
                        desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, (mel1, mel2, labels) in progress_bar:

        mel1 = mel1.to(device)
        mel2 = mel2.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        emb1 = model(mel1)
        emb2 = model(mel2)

        loss = criterion(emb1, emb2, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 🔹 Update tqdm live loss
        progress_bar.set_postfix(loss=loss.item())

        # 🔹 Extra debug print
        if batch_idx % print_every == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx}/{len(dataloader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(dataloader)
    print(f"\nEpoch [{epoch+1}/{num_epochs}] - Avg Loss: {avg_loss:.4f}\n")

Epoch 1/30:   0%|          | 3/3125 [00:01<20:39,  2.52it/s, loss=0.256]  

Epoch 1 | Batch 0/3125 | Loss: 0.3274


Epoch 1/30:   2%|▏         | 51/3125 [00:14<13:30,  3.79it/s, loss=0.291]

Epoch 1 | Batch 50/3125 | Loss: 0.2498


Epoch 1/30:   3%|▎         | 101/3125 [00:27<12:44,  3.95it/s, loss=0.223]

...

Epoch 30 | Batch 3050/3125 | Loss: 0.0129


Epoch 30/30:  99%|█████████▉| 3102/3125 [06:39<00:02,  7.93it/s, loss=0.0021]

Epoch 30 | Batch 3100/3125 | Loss: 0.0026


Epoch 30/30: 100%|██████████| 3125/3125 [06:42<00:00,  7.76it/s, loss=0.00284]


Epoch [30/30] - Avg Loss: 0.0060



## 13. Evaluation Function

`evaluate()` measures **speaker verification accuracy** on any DataLoader:
- Computes **cosine similarity** between the two speaker embeddings
- Predicts *same speaker* if similarity > 0.5
- Displays running accuracy live via `tqdm`
- No gradients computed (`torch.no_grad()` mode)

In [26]:
from tqdm import tqdm
import torch.nn.functional as F

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Evaluating")

    with torch.no_grad():
        for mel1, mel2, labels in progress_bar:

            mel1 = mel1.to(device)
            mel2 = mel2.to(device)
            labels = labels.to(device)

            emb1 = model(mel1)
            emb2 = model(mel2)

            cosine_sim = F.cosine_similarity(emb1, emb2)
            preds = (cosine_sim > 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # 🔥 Live accuracy update
            progress_bar.set_postfix(acc=correct / total)

    final_acc = correct / total
    print("\nFinal Accuracy:", final_acc)

## 14. Evaluate on Training Data

Run the trained model on the training set to measure in-sample accuracy as a quick sanity check.

In [27]:
evaluate(model, dataloader)

Evaluating: 100%|██████████| 3125/3125 [06:24<00:00,  8.13it/s, acc=0.945]


Final Accuracy: 0.94458


## 15. Save the Trained Model

Persist the model weights, current epoch, and optimizer state to `checkpoint.pth` on disk. This allows the model to be reloaded later for inference or training resumption without retraining.

In [28]:
torch.save({
    "model_state": model.state_dict(),
    "epoch": epoch,
    "optimizer_state": optimizer.state_dict()
}, "checkpoint.pth")

## 16. Load Test Dataset

Load the separate held-out test CSV. This dataset was not seen during training and provides an unbiased measure of the model's speaker verification performance.

In [29]:
TEST_CSV_PATH = "/kaggle/input/datasets/tahmidulislamomi/ml-project-testing-dataset/test_pairs.csv"

test_df = pd.read_csv(TEST_CSV_PATH)

print("Test rows:", len(test_df))
display(test_df.head())

Test rows: 50000


,audio_path_1,audio_path_2,label
0,test-clean/2961/961/2961-961-0008.flac,test-clean/2961/961/2961-961-0015.flac,1
1,test-clean/8555/284449/8555-284449-0015.flac,test-clean/8555/284447/8555-284447-0024.flac,1
2,test-clean/5639/40744/5639-40744-0007.flac,test-clean/5639/40744/5639-40744-0033.flac,1
3,test-clean/5683/32866/5683-32866-0026.flac,test-clean/5683/32879/5683-32879-0005.flac,1
4,test-clean/8230/279154/8230-279154-0002.flac,test-clean/8230/279154/8230-279154-0001.flac,1


## 17. Resolve Test Audio Paths

Follows the same pattern used for training: `to_test_absolute_path()` joins the relative path from the CSV with the real Kaggle test dataset directory.

In [30]:
import os

TEST_BASE_AUDIO_DIR = "/kaggle/input/datasets/tahmidulislamomi/ml-project-testing-dataset"

def to_test_absolute_path(rel_path):
    # Remove the hardcoded fake path
    cleaned = rel_path.replace("/kaggle/input/your-librispeech-dataset/", "", 1)
    # Join with your actual Kaggle directory
    return os.path.join(TEST_BASE_AUDIO_DIR, cleaned)



In [31]:
test_df["path1_abs"] = test_df["audio_path_1"].apply(to_test_absolute_path)
test_df["path2_abs"] = test_df["audio_path_2"].apply(to_test_absolute_path)

# Verify it works
print("Sample cleaned path:\n", test_df["path1_abs"].iloc[0])

Sample cleaned path:
 /kaggle/input/datasets/tahmidulislamomi/ml-project-testing-dataset/test-clean/2961/961/2961-961-0008.flac


## 18. Create Test Dataset

Instantiate `SpeakerPairDataset` for the test set using the same transforms (`mel_transform`, `amplitude_to_db`) as training.

In [32]:
test_dataset = SpeakerPairDataset(
    test_df,
    mel_transform,
    amplitude_to_db
)

## 19. Create Test DataLoader

Wrap the test dataset in a `DataLoader`.
- `shuffle = False` — preserve original row order for consistent, reproducible evaluation results

In [33]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,   # IMPORTANT: no shuffle for evaluation
    num_workers=2,
    pin_memory=True
)

## 20. Final Evaluation on Test Set

Run `evaluate()` on the held-out test set to obtain the **final test accuracy** — the key metric for this speaker verification project.

In [34]:
evaluate(model, test_loader)

Evaluating:   0%|          | 0/3125 [00:00<?, ?it/s]


TypeError: an integer is required